In [0]:
telemetry = spark.table("workspace.default.silver_telemetry")
maint = spark.table("workspace.default.silver_maint")
failures = spark.table("workspace.default.silver_failures")

print("Telemetry Schema:")
telemetry.printSchema()

print("Maintenance Schema:")
maint.printSchema()

print("Failures Schema:")
failures.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Maintenance Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- comp: string (nullable = true)

Failures Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)



In [0]:
print("Maintenance records:", maint.count())
print("Failure records:", failures.count())

print("\nMaintenance by component:")
maint.groupBy("comp").count().orderBy("comp").show()

print("Failures by component:")
failures.groupBy("failure").count().orderBy("failure").show()

Maintenance records: 3286
Failure records: 761

Maintenance by component:
+-----+-----+
| comp|count|
+-----+-----+
|comp1|  804|
|comp2|  863|
|comp3|  808|
|comp4|  811|
+-----+-----+

Failures by component:
+-------+-----+
|failure|count|
+-------+-----+
|  comp1|  192|
|  comp2|  259|
|  comp3|  131|
|  comp4|  179|
+-------+-----+



In [0]:
from pyspark.sql.functions import col, min

maintenance_to_failure = (
    maint.alias("m")
    .join(
        failures.alias("f"),
        (col("m.machineID") == col("f.machineID")) &
        (col("m.comp") == col("f.failure")) &
        (col("f.datetime") > col("m.datetime")),
        "left"
    )
    .groupBy(
        col("m.datetime").alias("maintenance_datetime"),
        col("m.machineID").alias("machineID"),
        col("m.comp").alias("component")
    )
    .agg(
        min("f.datetime").alias("next_failure_datetime")
    )
)

print("Maintenance events matched to next failure:",
      maintenance_to_failure.count())

maintenance_to_failure.show(10, truncate=False)

Maintenance events matched to next failure: 3286
+--------------------+---------+---------+---------------------+
|maintenance_datetime|machineID|component|next_failure_datetime|
+--------------------+---------+---------+---------------------+
|2015-02-04 06:00:00 |1        |comp4    |2015-06-19 06:00:00  |
|2014-11-28 06:00:00 |3        |comp2    |2015-01-07 06:00:00  |
|2015-07-01 06:00:00 |4        |comp1    |2015-08-30 06:00:00  |
|2015-01-19 06:00:00 |10       |comp2    |2015-04-04 06:00:00  |
|2015-02-19 06:00:00 |11       |comp3    |2015-04-20 06:00:00  |
|2015-07-10 06:00:00 |13       |comp2    |2015-08-09 06:00:00  |
|2014-10-14 06:00:00 |14       |comp2    |2015-01-31 06:00:00  |
|2015-05-31 06:00:00 |14       |comp1    |2015-07-15 06:00:00  |
|2014-06-01 06:00:00 |21       |comp3    |2015-01-23 06:00:00  |
|2015-07-07 06:00:00 |21       |comp2    |2015-08-06 06:00:00  |
+--------------------+---------+---------+---------------------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import datediff

rul_data = (
    maintenance_to_failure
    .withColumn(
        "rul_days",
        datediff(
            col("next_failure_datetime"),
            col("maintenance_datetime")
        )
    )
)

print("RUL records:", rul_data.count())

rul_data.select(
    "maintenance_datetime",
    "machineID",
    "component",
    "next_failure_datetime",
    "rul_days"
).show(10, truncate=False)

RUL records: 3286
+--------------------+---------+---------+---------------------+--------+
|maintenance_datetime|machineID|component|next_failure_datetime|rul_days|
+--------------------+---------+---------+---------------------+--------+
|2015-02-04 06:00:00 |1        |comp4    |2015-06-19 06:00:00  |135     |
|2014-11-28 06:00:00 |3        |comp2    |2015-01-07 06:00:00  |40      |
|2015-07-01 06:00:00 |4        |comp1    |2015-08-30 06:00:00  |60      |
|2015-01-19 06:00:00 |10       |comp2    |2015-04-04 06:00:00  |75      |
|2015-02-19 06:00:00 |11       |comp3    |2015-04-20 06:00:00  |60      |
|2015-07-10 06:00:00 |13       |comp2    |2015-08-09 06:00:00  |30      |
|2014-10-14 06:00:00 |14       |comp2    |2015-01-31 06:00:00  |109     |
|2015-05-31 06:00:00 |14       |comp1    |2015-07-15 06:00:00  |45      |
|2014-06-01 06:00:00 |21       |comp3    |2015-01-23 06:00:00  |236     |
|2015-07-07 06:00:00 |21       |comp2    |2015-08-06 06:00:00  |30      |
+-------------------

In [0]:
from pyspark.sql.functions import col, count, when

print("Maintenance events with known next failure:")
print(
    rul_data
    .filter(col("next_failure_datetime").isNotNull())
    .count()
)

print("\nMaintenance events without a known next failure:")
print(
    rul_data
    .filter(col("next_failure_datetime").isNull())
    .count()
)

print("\nRUL statistics:")
rul_data.filter(
    col("rul_days").isNotNull()
).select("rul_days").describe().show()

Maintenance events with known next failure:
1432

Maintenance events without a known next failure:
1854

RUL statistics:
+-------+------------------+
|summary|          rul_days|
+-------+------------------+
|  count|              1432|
|   mean|113.24371508379889|
| stddev| 76.98992983891335|
|    min|                 1|
|    max|               488|
+-------+------------------+



In [0]:
from pyspark.sql.functions import avg, stddev

rul_known = (
    rul_data
    .filter(col("next_failure_datetime").isNotNull())
)

telemetry_rul = (
    rul_known.alias("r")
    .join(
        telemetry.alias("t"),
        (col("r.machineID") == col("t.machineID")) &
        (col("t.datetime") >= col("r.maintenance_datetime")) &
        (col("t.datetime") < col("r.next_failure_datetime")),
        "inner"
    )
    .groupBy(
        col("r.maintenance_datetime"),
        col("r.machineID"),
        col("r.component"),
        col("r.next_failure_datetime"),
        col("r.rul_days")
    )
    .agg(
        avg("t.volt").alias("avg_volt"),
        stddev("t.volt").alias("std_volt"),
        avg("t.rotate").alias("avg_rotate"),
        stddev("t.rotate").alias("std_rotate"),
        avg("t.pressure").alias("avg_pressure"),
        stddev("t.pressure").alias("std_pressure"),
        avg("t.vibration").alias("avg_vibration"),
        stddev("t.vibration").alias("std_vibration")
    )
)

print("RUL records with telemetry features:", telemetry_rul.count())

telemetry_rul.printSchema()

telemetry_rul.show(10, truncate=False)

RUL records with telemetry features: 1432
root
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- next_failure_datetime: timestamp (nullable = true)
 |-- rul_days: integer (nullable = true)
 |-- avg_volt: double (nullable = true)
 |-- std_volt: double (nullable = true)
 |-- avg_rotate: double (nullable = true)
 |-- std_rotate: double (nullable = true)
 |-- avg_pressure: double (nullable = true)
 |-- std_pressure: double (nullable = true)
 |-- avg_vibration: double (nullable = true)
 |-- std_vibration: double (nullable = true)

+--------------------+---------+---------+---------------------+--------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|maintenance_datetime|machineID|component|next_failure_datetime|rul_days|avg_volt          |std_volt          |avg_rotate        |std_rotate        |

In [0]:
model_data = (
    telemetry_rul
    .select(
        "machineID",
        "component",
        "avg_volt",
        "std_volt",
        "avg_rotate",
        "std_rotate",
        "avg_pressure",
        "std_pressure",
        "avg_vibration",
        "std_vibration",
        "rul_days"
    )
    .dropna()
)

print("Regression records:", model_data.count())

model_data.printSchema()

model_data.show(10, truncate=False)

Regression records: 1432
root
 |-- machineID: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- avg_volt: double (nullable = true)
 |-- std_volt: double (nullable = true)
 |-- avg_rotate: double (nullable = true)
 |-- std_rotate: double (nullable = true)
 |-- avg_pressure: double (nullable = true)
 |-- std_pressure: double (nullable = true)
 |-- avg_vibration: double (nullable = true)
 |-- std_vibration: double (nullable = true)
 |-- rul_days: integer (nullable = true)

+---------+---------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------+
|machineID|component|avg_volt          |std_volt          |avg_rotate        |std_rotate        |avg_pressure      |std_pressure      |avg_vibration     |std_vibration     |rul_days|
+---------+---------+------------------+------------------+------------------+------------------+------------------+------------------+--

In [0]:
from sklearn.model_selection import train_test_split

model_pd = model_data.toPandas()

feature_columns_q8 = [
    "avg_volt",
    "std_volt",
    "avg_rotate",
    "std_rotate",
    "avg_pressure",
    "std_pressure",
    "avg_vibration",
    "std_vibration"
]

X = model_pd[feature_columns_q8]
y = model_pd["rul_days"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training records: 1145
Testing records: 287
Training features: (1145, 8)
Testing features: (287, 8)


In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

model_q8 = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model_q8.fit(X_train, y_train)

y_pred = model_q8.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RUL Regression Model Results")
print("--------------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

RUL Regression Model Results
--------------------------------
MAE : 43.114090639506216
RMSE: 60.33723274862636
R²  : 0.3961791629571878


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, avg, stddev, max

# Latest maintenance for each machine and component
window_maint = Window.partitionBy("machineID", "comp").orderBy(
    col("datetime").desc()
)

latest_maint = (
    maint
    .withColumn("rn", row_number().over(window_maint))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Latest maintenance records:", latest_maint.count())

latest_maint.show(10, truncate=False)

Latest maintenance records: 400
+-------------------+---------+-----+
|datetime           |machineID|comp |
+-------------------+---------+-----+
|2015-12-31 06:00:00|1        |comp1|
|2015-11-16 06:00:00|1        |comp2|
|2015-12-01 06:00:00|1        |comp3|
|2015-12-16 06:00:00|1        |comp4|
|2015-12-14 06:00:00|2        |comp1|
|2015-12-29 06:00:00|2        |comp2|
|2015-11-29 06:00:00|2        |comp3|
|2015-10-30 06:00:00|2        |comp4|
|2015-11-18 06:00:00|3        |comp1|
|2015-12-18 06:00:00|3        |comp2|
+-------------------+---------+-----+
only showing top 10 rows


In [0]:
current_telemetry = (
    latest_maint.alias("m")
    .join(
        telemetry.alias("t"),
        (col("m.machineID") == col("t.machineID")) &
        (col("t.datetime") >= col("m.datetime")),
        "inner"
    )
    .groupBy(
        col("m.machineID").alias("machineID"),
        col("m.comp").alias("component"),
        col("m.datetime").alias("maintenance_datetime")
    )
    .agg(
        avg("t.volt").alias("avg_volt"),
        stddev("t.volt").alias("std_volt"),
        avg("t.rotate").alias("avg_rotate"),
        stddev("t.rotate").alias("std_rotate"),
        avg("t.pressure").alias("avg_pressure"),
        stddev("t.pressure").alias("std_pressure"),
        avg("t.vibration").alias("avg_vibration"),
        stddev("t.vibration").alias("std_vibration"),
        max("t.datetime").alias("current_datetime")
    )
    .dropna()
)

print("Current active component records:", current_telemetry.count())

current_telemetry.printSchema()

current_telemetry.show(10, truncate=False)

Current active component records: 393
root
 |-- machineID: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- avg_volt: double (nullable = true)
 |-- std_volt: double (nullable = true)
 |-- avg_rotate: double (nullable = true)
 |-- std_rotate: double (nullable = true)
 |-- avg_pressure: double (nullable = true)
 |-- std_pressure: double (nullable = true)
 |-- avg_vibration: double (nullable = true)
 |-- std_vibration: double (nullable = true)
 |-- current_datetime: timestamp (nullable = true)

+---------+---------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+
|machineID|component|maintenance_datetime|avg_volt          |std_volt          |avg_rotate        |std_rotate        |avg_pressure      |std_pressure      |avg_vibration     |std_vibration     |current_datetime

In [0]:
current_pd = current_telemetry.toPandas()

X_current = current_pd[feature_columns_q8]

current_pd["predicted_rul_days"] = model_q8.predict(X_current)

current_pd["predicted_rul_days"] = (
    current_pd["predicted_rul_days"]
    .clip(lower=0)
)

print("Predictions generated:", len(current_pd))

print("\nPredicted RUL summary:")
print(current_pd["predicted_rul_days"].describe())

print("\nComponents with lowest predicted RUL:")
print(
    current_pd[
        ["machineID", "component", "maintenance_datetime",
         "current_datetime", "predicted_rul_days"]
    ]
    .sort_values("predicted_rul_days")
    .head(10)
    .to_string(index=False)
)

Predictions generated: 393

Predicted RUL summary:
count    393.000000
mean      99.888720
std       26.833109
min       48.203588
25%       78.906447
50%       96.637296
75%      113.406663
max      213.548818
Name: predicted_rul_days, dtype: float64

Components with lowest predicted RUL:
 machineID component maintenance_datetime    current_datetime  predicted_rul_days
        90     comp3  2015-11-16 06:00:00 2016-01-01 06:00:00           48.203588
         7     comp4  2015-12-05 06:00:00 2016-01-01 06:00:00           53.435183
        86     comp1  2015-11-04 06:00:00 2016-01-01 06:00:00           54.807078
        68     comp4  2015-11-28 06:00:00 2016-01-01 06:00:00           55.347757
        68     comp1  2015-11-28 06:00:00 2016-01-01 06:00:00           55.347757
        85     comp3  2015-12-11 06:00:00 2016-01-01 06:00:00           55.651504
        13     comp3  2015-11-22 06:00:00 2016-01-01 06:00:00           56.166093
        29     comp2  2015-12-13 06:00:00 2016-01-01 

In [0]:
from pyspark.sql.functions import lit

gold_q8 = spark.createDataFrame(current_pd)

gold_q8 = gold_q8.select(
    "machineID",
    "component",
    "maintenance_datetime",
    "current_datetime",
    "avg_volt",
    "std_volt",
    "avg_rotate",
    "std_rotate",
    "avg_pressure",
    "std_pressure",
    "avg_vibration",
    "std_vibration",
    "predicted_rul_days"
)

gold_q8.printSchema()

gold_q8.show(10, truncate=False)

root
 |-- machineID: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- current_datetime: timestamp (nullable = true)
 |-- avg_volt: double (nullable = true)
 |-- std_volt: double (nullable = true)
 |-- avg_rotate: double (nullable = true)
 |-- std_rotate: double (nullable = true)
 |-- avg_pressure: double (nullable = true)
 |-- std_pressure: double (nullable = true)
 |-- avg_vibration: double (nullable = true)
 |-- std_vibration: double (nullable = true)
 |-- predicted_rul_days: double (nullable = true)

+---------+---------+--------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|machineID|component|maintenance_datetime|current_datetime   |avg_volt          |std_volt          |avg_rotate        |std_rotate        |avg_pressure      |std_pressure      |avg

In [0]:
from delta.tables import DeltaTable

gold_table_name = "workspace.default.gold_q8_rul_prediction"

if not spark.catalog.tableExists(gold_table_name):

    gold_q8.write \
        .format("delta") \
        .saveAsTable(gold_table_name)

    print("Gold Q8 RUL table created")

else:

    target = DeltaTable.forName(
        spark,
        gold_table_name
    )

    target.alias("t").merge(
        gold_q8.alias("s"),
        """
        t.machineID = s.machineID
        AND t.component = s.component
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print("Gold Q8 RUL table updated")

print("Gold table row count:", spark.table(gold_table_name).count())

Gold Q8 RUL table created
Gold table row count: 393


In [0]:
from pyspark.sql.functions import concat, lit, col

display(
    spark.table("workspace.default.gold_q8_rul_prediction")
    .orderBy("predicted_rul_days")
    .limit(10)
    .withColumn(
        "machine_component",
        concat(
            lit("Machine "),
            col("machineID"),
            lit(" - "),
            col("component")
        )
    )
    .select(
        "machine_component",
        "predicted_rul_days"
    )
)

machine_component,predicted_rul_days
Machine 90 - comp3,48.20358755830009
Machine 7 - comp4,53.43518332478422
Machine 86 - comp1,54.807078110781816
Machine 68 - comp1,55.34775678054805
Machine 68 - comp4,55.34775678054805
Machine 85 - comp3,55.65150408450408
Machine 13 - comp3,56.16609253940503
Machine 29 - comp2,56.86357465436442
Machine 29 - comp4,56.86357465436442
Machine 90 - comp2,57.73086845793095


Databricks visualization. Run in Databricks to view.